# Build the ORDEM Pointing Library

Parses the initial ORDEM pointing jobs into a response matrix and audits which elevation and azimuth choices provide distinct information.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, re, shutil, tempfile, time, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


RAW_DIR = PROJECT_ROOT / 'ordem_raw'
BATCH_NAME = 'job-33aa70ca-8ffd-4610-bd7d-b354c4f5a765.zip'
BASELINE_NAME = 'job-d80e4122-e27b-4ec6-903f-8e06e33ca540.zip'
BATCH_ZIP = RAW_DIR / BATCH_NAME
BASELINE_ZIP = RAW_DIR / BASELINE_NAME

EXPECTED_BATCH_SHA256 = '4ca5e4b98d55c8e41229c8865d0a66364d1fd0cfb230381fb46fb411c7380953'
EXPECTED_BASELINE_SHA256 = 'b70d174128a5bfb3f1f847413e4afa3eea24dcc31635cdd02ba1075b4ca9f588'

OUTPUT_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / '33aa70ca_80_plus_baseline'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RADAR_RANGE_MIN_KM = 1026.639
RADAR_RANGE_MAX_KM = 1084.050
EXPECTED_AZ = np.arange(80.0, 100.0 + 0.01, 2.5)
EXPECTED_EL = np.arange(65.0, 85.0 + 0.01, 2.5)
BASELINE_POINTING = (90.0, 75.0)

THRESHOLDS = [
    ('10', 1e-5, '10um', '10 µm'),
    ('15', 10**-4.5, '31p6um', '31.6 µm'),
    ('20', 1e-4, '100um', '100 µm'),
    ('25', 10**-3.5, '316um', '316 µm'),
    ('30', 1e-3, '1mm', '1 mm'),
    ('35', 10**-2.5, '3p16mm', '3.16 mm'),
    ('40', 1e-2, '1cm', '1 cm'),
    ('45', 10**-1.5, '3p16cm', '3.16 cm'),
    ('50', 1e-1, '10cm', '10 cm'),
    ('55', 10**-0.5, '31p6cm', '31.6 cm'),
    ('60', 1.0, '1m', '1 m'),
]

print('research root:', PROJECT_ROOT)
print('Batch input:', BATCH_ZIP)
print('Center input:', BASELINE_ZIP)
print('Output:', OUTPUT_DIR)
print('CPU runtime is sufficient; no package installation is needed.')


## 1. Helpers: safe extraction, checksums, metadata, and ORDEM table parser


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        zf.extractall(destination)


def result_directories(root):
    return sorted({p.parent for p in Path(root).rglob('ORDEM.IN')})


def parse_ordem_metadata(result_dir):
    result_dir = Path(result_dir)
    lines = (result_dir / 'ORDEM.IN').read_text(errors='replace').splitlines()
    values = {}
    for line in lines:
        if '!' not in line:
            continue
        raw, comment = line.split('!', 1)
        raw, comment = raw.strip(), comment.strip().lower()
        if not raw:
            continue
        if 'year of observation' in comment:
            values['year'] = int(float(raw))
        elif 'sensor latitude' in comment:
            values['sensor_lat_deg'] = float(raw)
        elif comment.startswith('azimuth'):
            values['azimuth_deg'] = float(raw)
        elif comment.startswith('elevation'):
            values['elevation_deg'] = float(raw)
        elif 'file defining all igloo element boundaries (alt)' in comment:
            values['igloo_file'] = raw

    flux_header = ''
    with open(result_dir / 'IGLOOFLUX_TEL.OUT', errors='replace') as f:
        for line in f:
            if 'Sensor lat.' in line and 'Pointing AZ' in line:
                flux_header = line
                break
    match = re.search(
        r'Year:\s*(\d+).*Elements:\s*(\d+).*Populations:\s*(\d+).*'
        r'Sensor lat\.\s*=\s*([-+0-9.]+).*Pointing AZ\s*=\s*([-+0-9.]+).*'
        r'Pointing EL\s*=\s*([-+0-9.]+)',
        flux_header,
    )
    if not match:
        raise ValueError(f'Could not parse IGLOOFLUX header in {result_dir}')
    values.update({
        'year_header': int(match.group(1)),
        'n_elements_header': int(match.group(2)),
        'n_populations_header': int(match.group(3)),
        'sensor_lat_header_deg': float(match.group(4)),
        'azimuth_header_deg': float(match.group(5)),
        'elevation_header_deg': float(match.group(6)),
    })
    values['job_key'] = result_dir.name
    values['complete'] = 'Complete' in (result_dir / 'status.txt').read_text(errors='replace')
    return values


def read_ordem_table(path):
    path = Path(path)
    lines = path.read_text(errors='replace').splitlines()
    header_idx = next(i for i, line in enumerate(lines) if line.lstrip().startswith('Element'))
    columns = lines[header_idx].split()
    rows = []
    for line in lines[header_idx + 1:]:
        fields = line.split()
        if len(fields) != len(columns):
            continue
        try:
            rows.append([float(x) for x in fields])
        except ValueError:
            continue
    frame = pd.DataFrame(rows, columns=columns)
    if frame.empty:
        raise ValueError(f'No numeric table parsed from {path}')
    frame['Element'] = frame['Element'].astype(int)
    return frame


def action_id(azimuth, elevation):
    def token(x):
        return f'{x:05.1f}'.replace('.', 'p')
    return f'az{token(azimuth)}_el{token(elevation)}'


def suffix_from_column(column):
    return column.split('_', 1)[1]


def overlap_fraction(low, high, window_low, window_high):
    width = high - low
    overlap = np.maximum(0.0, np.minimum(high, window_high) - np.maximum(low, window_low))
    return np.divide(overlap, width, out=np.zeros_like(overlap, dtype=float), where=width > 0)


## 2. Locate, checksum, extract, and audit all 81 pointing jobs


In [ ]:
for path in (BATCH_ZIP, BASELINE_ZIP):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

batch_hash = sha256_file(BATCH_ZIP)
baseline_hash = sha256_file(BASELINE_ZIP)
print('80-job SHA256:', batch_hash)
print('center SHA256:', baseline_hash)
if batch_hash != EXPECTED_BATCH_SHA256:
    raise RuntimeError('The 80-job ZIP checksum differs from the audited archive. Stop and inspect it.')
if baseline_hash != EXPECTED_BASELINE_SHA256:
    raise RuntimeError('The center ZIP checksum differs from the audited archive. Stop and inspect it.')

work_root = Path(tempfile.mkdtemp(prefix='debris_ordem_action_'))
batch_root = work_root / 'batch80'
baseline_root = work_root / 'baseline'
safe_extract(BATCH_ZIP, batch_root)
safe_extract(BASELINE_ZIP, baseline_root)

batch_dirs = result_directories(batch_root)
baseline_dirs = result_directories(baseline_root)
if len(batch_dirs) != 80:
    raise RuntimeError(f'Expected 80 batch result folders, found {len(batch_dirs)}')
if len(baseline_dirs) != 1:
    raise RuntimeError(f'Expected 1 center result folder, found {len(baseline_dirs)}')

batch_required = {'ORDEM.IN', 'ORDEM-Proj.prj', 'IGLOOFLUX_TEL.OUT', 'FLUX_TEL.OUT', 'status.txt'}
baseline_required = batch_required | {'IGLOOFLUX_SIGMARAN_TEL.OUT', 'IGLOOFLUX_SIGMAPOP_TEL.OUT'}

inventory_rows = []
for source, directories, required in [
    ('batch80', batch_dirs, batch_required),
    ('center_baseline', baseline_dirs, baseline_required),
]:
    for directory in directories:
        missing = sorted(name for name in required if not (directory / name).is_file())
        if missing:
            raise RuntimeError(f'{directory.name} is missing {missing}')
        meta = parse_ordem_metadata(directory)
        meta.update({
            'source': source,
            'result_directory': str(directory),
            'has_sigmaran': (directory / 'IGLOOFLUX_SIGMARAN_TEL.OUT').is_file(),
            'has_sigmapop': (directory / 'IGLOOFLUX_SIGMAPOP_TEL.OUT').is_file(),
        })
        if not meta['complete']:
            raise RuntimeError(f'ORDEM job is not marked Complete: {directory.name}')
        if meta['year'] != meta['year_header']:
            raise RuntimeError(f'Year mismatch in {directory.name}')
        if not np.isclose(meta['azimuth_deg'], meta['azimuth_header_deg']):
            raise RuntimeError(f'Azimuth mismatch in {directory.name}')
        if not np.isclose(meta['elevation_deg'], meta['elevation_header_deg']):
            raise RuntimeError(f'Elevation mismatch in {directory.name}')
        meta['action_id'] = action_id(meta['azimuth_header_deg'], meta['elevation_header_deg'])
        inventory_rows.append(meta)

inventory = pd.DataFrame(inventory_rows)
inventory['azimuth_deg'] = inventory['azimuth_header_deg']
inventory['elevation_deg'] = inventory['elevation_header_deg']
inventory = inventory.sort_values(['elevation_deg', 'azimuth_deg']).reset_index(drop=True)

batch_coords = set(map(tuple, inventory.loc[inventory.source == 'batch80', ['azimuth_deg', 'elevation_deg']].round(6).to_numpy()))
expected_coords = {(round(float(a), 6), round(float(e), 6)) for a in EXPECTED_AZ for e in EXPECTED_EL}
expected_coords.remove(BASELINE_POINTING)
if batch_coords != expected_coords:
    missing = sorted(expected_coords - batch_coords)
    extra = sorted(batch_coords - expected_coords)
    raise RuntimeError(f'Pointing grid mismatch. Missing={missing}; extra={extra}')

if inventory['action_id'].duplicated().any():
    raise RuntimeError('Duplicate action IDs found after adding the center job.')
if len(inventory) != 81:
    raise RuntimeError(f'Expected 81 total actions, found {len(inventory)}')
if set(inventory['year']) != {2018}:
    raise RuntimeError(f'Unexpected years: {sorted(inventory.year.unique())}')
if set(inventory['igloo_file']) != {'ALT_5_LEO.TIG'}:
    raise RuntimeError(f'Unexpected igloo files: {sorted(inventory.igloo_file.unique())}')

print('AUDIT PASSED')
print('  Completed batch jobs:', len(batch_dirs))
print('  Center baseline jobs:', len(baseline_dirs))
print('  Total unique actions:', len(inventory))
print('  Azimuths:', sorted(inventory.azimuth_deg.unique()))
print('  Elevations:', sorted(inventory.elevation_deg.unique()))
print('  Batch uncertainty files:', int(inventory.loc[inventory.source == 'batch80', 'has_sigmaran'].sum()), 'of 80')
print('  Center uncertainty files:', bool(inventory.loc[inventory.source == 'center_baseline', 'has_sigmaran'].iloc[0]))


## 3. Parse the directional fluxes and build the radar-window response matrix


In [ ]:
started = time.time()
wide_frames = []
response_frames = []
reference_altitude_grid = None

for index, row in inventory.iterrows():
    directory = Path(row['result_directory'])
    flux = read_ordem_table(directory / 'IGLOOFLUX_TEL.OUT')
    id_columns = ['Element', 'alt_low', 'alt_high', 'rng_low', 'rng_high']
    flux_columns = [c for c in flux.columns if c.startswith('Flux_')]
    if len(flux) != 360 or len(flux_columns) != 55:
        raise RuntimeError(f"{row['action_id']}: expected 360 × 55 flux table, got {len(flux)} × {len(flux_columns)}")
    if not np.isfinite(flux[flux_columns].to_numpy()).all():
        raise RuntimeError(f"{row['action_id']}: non-finite flux value")
    if (flux[flux_columns].to_numpy() < 0).any():
        raise RuntimeError(f"{row['action_id']}: negative flux value")

    altitude_grid = flux[['Element', 'alt_low', 'alt_high']].to_numpy()
    if reference_altitude_grid is None:
        reference_altitude_grid = altitude_grid.copy()
    elif not np.allclose(altitude_grid, reference_altitude_grid, rtol=0, atol=1e-9):
        raise RuntimeError(f"{row['action_id']}: altitude grid differs from the other actions")

    wide = flux.copy()
    wide.insert(0, 'source', row['source'])
    wide.insert(0, 'elevation_deg', row['elevation_deg'])
    wide.insert(0, 'azimuth_deg', row['azimuth_deg'])
    wide.insert(0, 'action_id', row['action_id'])
    wide_frames.append(wide)

    response = flux[id_columns].copy()
    response.insert(0, 'source', row['source'])
    response.insert(0, 'elevation_deg', row['elevation_deg'])
    response.insert(0, 'azimuth_deg', row['azimuth_deg'])
    response.insert(0, 'action_id', row['action_id'])
    response['range_overlap_fraction'] = overlap_fraction(
        response['rng_low'].to_numpy(), response['rng_high'].to_numpy(),
        RADAR_RANGE_MIN_KM, RADAR_RANGE_MAX_KM,
    )
    response['range_overlap_km'] = response['range_overlap_fraction'] * (response['rng_high'] - response['rng_low'])
    for code_value, diameter_m, key, display_label in THRESHOLDS:
        selected = [c for c in flux_columns if c.endswith(code_value)]
        if len(selected) != 5:
            raise RuntimeError(f"{row['action_id']}: expected 5 material populations for code {code_value}, found {selected}")
        cumulative = flux[selected].sum(axis=1)
        response[f'flux_gt_{key}_m2_yr'] = cumulative
        response[f'window_contribution_gt_{key}_m2_yr'] = cumulative * response['range_overlap_fraction']
    response_frames.append(response)

    elapsed = time.time() - started
    done = index + 1
    eta = elapsed / done * (len(inventory) - done)
    print(f"[{done:02d}/81] {row['action_id']} parsed; ETA {eta:5.1f} s")

wide_all = pd.concat(wide_frames, ignore_index=True)
response_all = pd.concat(response_frames, ignore_index=True)
print('Parsed wide rows:', f'{len(wide_all):,}')
print('Response rows:', f'{len(response_all):,}')


## 4. Build the action summary and preserve the center uncertainty tables


In [ ]:
summary_rows = []
for action, group in response_all.groupby('action_id', sort=False):
    base = group.iloc[0]
    active = group['range_overlap_fraction'] > 0
    item = {
        'action_id': action,
        'source': base['source'],
        'azimuth_deg': float(base['azimuth_deg']),
        'elevation_deg': float(base['elevation_deg']),
        'overlap_bin_count': int(active.sum()),
        'overlap_span_km': float(group['range_overlap_km'].sum()),
        'overlap_alt_min_km': float(group.loc[active, 'alt_low'].min()),
        'overlap_alt_max_km': float(group.loc[active, 'alt_high'].max()),
    }
    for code_value, diameter_m, key, display_label in THRESHOLDS:
        item[f'full_flux_gt_{key}_m2_yr'] = float(group[f'flux_gt_{key}_m2_yr'].sum())
        item[f'window_flux_gt_{key}_m2_yr'] = float(group[f'window_contribution_gt_{key}_m2_yr'].sum())
    summary_rows.append(item)

action_summary = pd.DataFrame(summary_rows).sort_values(['elevation_deg', 'azimuth_deg']).reset_index(drop=True)
center_mask = np.isclose(action_summary.azimuth_deg, 90.0) & np.isclose(action_summary.elevation_deg, 75.0)
if center_mask.sum() != 1:
    raise RuntimeError('Could not identify exactly one 90°/75° center action')
center = action_summary.loc[center_mask].iloc[0]
for _, _, key, _ in THRESHOLDS:
    denom = float(center[f'window_flux_gt_{key}_m2_yr'])
    action_summary[f'window_ratio_to_center_gt_{key}'] = action_summary[f'window_flux_gt_{key}_m2_yr'] / denom if denom > 0 else np.nan

center_dir = Path(inventory.loc[inventory.source == 'center_baseline', 'result_directory'].iloc[0])
center_flux = read_ordem_table(center_dir / 'IGLOOFLUX_TEL.OUT')
center_ran = read_ordem_table(center_dir / 'IGLOOFLUX_SIGMARAN_TEL.OUT')
center_pop = read_ordem_table(center_dir / 'IGLOOFLUX_SIGMAPOP_TEL.OUT')
id_columns = ['Element', 'alt_low', 'alt_high', 'rng_low', 'rng_high']
if not np.allclose(center_flux[id_columns], center_ran[id_columns]) or not np.allclose(center_flux[id_columns], center_pop[id_columns]):
    raise RuntimeError('Center flux and uncertainty grids do not align')

uncertainty_rows = []
for flux_column in [c for c in center_flux if c.startswith('Flux_')]:
    suffix = suffix_from_column(flux_column)
    ran_column = f'Sig(ran)_{suffix}'
    pop_column = f'Sig(pop)_{suffix}'
    values = center_flux[flux_column].to_numpy()
    ran = center_ran[ran_column].to_numpy()
    pop = center_pop[pop_column].to_numpy()
    quadrature = np.hypot(ran, pop)
    relative = np.divide(quadrature, values, out=np.full_like(values, np.nan), where=values > 0)
    material = suffix[:2]
    size_code = suffix[2:]
    threshold = next(t for t in THRESHOLDS if t[0] == size_code)
    for i in range(len(center_flux)):
        uncertainty_rows.append({
            'action_id': action_id(90.0, 75.0),
            'Element': int(center_flux.loc[i, 'Element']),
            'alt_low': float(center_flux.loc[i, 'alt_low']),
            'alt_high': float(center_flux.loc[i, 'alt_high']),
            'rng_low': float(center_flux.loc[i, 'rng_low']),
            'rng_high': float(center_flux.loc[i, 'rng_high']),
            'material': material,
            'size_code': size_code,
            'diameter_m': threshold[1],
            'diameter_label': threshold[3],
            'flux_m2_yr': float(values[i]),
            'sigma_range_m2_yr': float(ran[i]),
            'sigma_population_m2_yr': float(pop[i]),
            'sigma_quadrature_m2_yr': float(quadrature[i]),
            'relative_sigma_quadrature': float(relative[i]) if np.isfinite(relative[i]) else np.nan,
            'uncertainty_scope': 'NASA ORDEM center job only (AZ=90, EL=75)',
        })
center_uncertainty = pd.DataFrame(uncertainty_rows)

variation_rows = []
for _, diameter_m, key, display_label in THRESHOLDS:
    values = action_summary[f'window_flux_gt_{key}_m2_yr'].to_numpy()
    variation_rows.append({
        'diameter_m': diameter_m,
        'diameter_label': display_label,
        'mean_window_flux_m2_yr': float(values.mean()),
        'sd_across_actions_m2_yr': float(values.std(ddof=1)),
        'coefficient_of_variation': float(values.std(ddof=1) / values.mean()) if values.mean() > 0 else np.nan,
        'minimum_to_maximum_ratio': float(values.min() / values.max()) if values.max() > 0 else np.nan,
        'max_to_min_ratio': float(values.max() / values.min()) if values.min() > 0 else np.inf,
    })
directional_variation = pd.DataFrame(variation_rows)

print('Center-only uncertainty rows:', f'{len(center_uncertainty):,}')
print('Action summary rows:', len(action_summary))
display(action_summary[['action_id', 'window_flux_gt_1mm_m2_yr', 'window_ratio_to_center_gt_1mm']]
        .sort_values('window_flux_gt_1mm_m2_yr', ascending=False).head(10))


## 5. Save the reproducible action library


In [ ]:
inventory_out = inventory.drop(columns=['result_directory']).copy()
inventory_out.to_csv(OUTPUT_DIR / 'ordem_action_inventory.csv', index=False)
wide_all.to_csv(OUTPUT_DIR / 'ordem_flux_wide_all_actions.csv.gz', index=False, compression='gzip')
response_all.to_csv(OUTPUT_DIR / 'ordem_action_response_matrix.csv.gz', index=False, compression='gzip')
action_summary.to_csv(OUTPUT_DIR / 'ordem_radar_window_action_summary.csv', index=False)
center_uncertainty.to_csv(OUTPUT_DIR / 'ordem_center_uncertainty_long.csv.gz', index=False, compression='gzip')
directional_variation.to_csv(OUTPUT_DIR / 'ordem_directional_variation_by_size.csv', index=False)

for _, _, key, _ in THRESHOLDS:
    pivot = action_summary.pivot(index='elevation_deg', columns='azimuth_deg', values=f'window_flux_gt_{key}_m2_yr')
    pivot.sort_index(ascending=True).to_csv(OUTPUT_DIR / f'ordem_window_flux_grid_gt_{key}.csv')

print('Saved tabular action library to:', OUTPUT_DIR)


## 6. Directional heatmaps and size-dependent action diversity


In [ ]:
plt.rcParams.update({'font.size': 10, 'figure.dpi': 130})
fig, axes = plt.subplots(2, 2, figsize=(13.2, 10.2), constrained_layout=True)
heatmap_specs = [('1mm', '>1 mm'), ('1cm', '>1 cm'), ('10cm', '>10 cm')]

for ax, (key, display_label) in zip(axes.flat[:3], heatmap_specs):
    pivot = action_summary.pivot(index='elevation_deg', columns='azimuth_deg', values=f'window_ratio_to_center_gt_{key}')
    pivot = pivot.sort_index(ascending=True).sort_index(axis=1)
    values = pivot.to_numpy()
    finite = values[np.isfinite(values)]
    vmin, vmax = float(finite.min()), float(finite.max())
    if vmin < 1.0 < vmax:
        norm = TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)
        image = ax.imshow(values, origin='lower', aspect='auto', cmap='coolwarm', norm=norm)
    else:
        image = ax.imshow(values, origin='lower', aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(pivot.columns)), [f'{x:g}' for x in pivot.columns], rotation=45, ha='right')
    ax.set_yticks(range(len(pivot.index)), [f'{x:g}' for x in pivot.index])
    ax.set_xlabel('Azimuth (deg)')
    ax.set_ylabel('Elevation (deg)')
    ax.set_title(f'ORDEM radar-window flux ratio, {display_label}\n(relative to AZ 90°, EL 75°)')
    center_x = list(pivot.columns).index(90.0)
    center_y = list(pivot.index).index(75.0)
    ax.scatter(center_x, center_y, marker='*', s=170, facecolor='white', edgecolor='black', linewidth=0.8)
    cb = fig.colorbar(image, ax=ax, shrink=0.82)
    cb.set_label('Flux ratio to center')

ax = axes.flat[3]
ax.plot(directional_variation['diameter_m'], 100 * directional_variation['coefficient_of_variation'], marker='o', lw=2)
ax.set_xscale('log')
ax.set_xlabel('Minimum object diameter (m)')
ax.set_ylabel('Across-action coefficient of variation (%)')
ax.set_title('How much pointing changes the ORDEM response')
ax.grid(True, which='both', alpha=0.25)
for _, row in directional_variation.iloc[[0, 4, 6, 8, 10]].iterrows():
    ax.annotate(row['diameter_label'], (row['diameter_m'], 100 * row['coefficient_of_variation']),
                textcoords='offset points', xytext=(3, 5), fontsize=8)

fig.suptitle('ORDEM 3.2 local pointing action library — Tromsø, 2018', fontsize=15)
fig.savefig(OUTPUT_DIR / 'ordem_action_library_diagnostics_2x2.png', bbox_inches='tight', dpi=220)
fig.savefig(OUTPUT_DIR / 'ordem_action_library_diagnostics_2x2.pdf', bbox_inches='tight')
plt.show()


## 7. Final QA, manifest, and success marker


In [ ]:
expected_rows = 81 * 360
checks = {'81_unique_actions': len(inventory) == 81 and inventory.action_id.nunique() == 81, '80_batch_plus_center': int((inventory.source == 'batch80').sum()) == 80 and int((inventory.source == 'center_baseline').sum()) == 1, 'all_jobs_complete': bool(inventory.complete.all()), 'wide_row_count': len(wide_all) == expected_rows, 'response_row_count': len(response_all) == expected_rows, 'no_negative_flux': bool((wide_all.filter(regex='^Flux_') >= 0).to_numpy().all()), 'no_nonfinite_flux': bool(np.isfinite(wide_all.filter(regex='^Flux_').to_numpy()).all()), 'center_uncertainty_present': len(center_uncertainty) == 360 * 55, 'batch_uncertainty_not_invented': int(inventory.loc[inventory.source == 'batch80', 'has_sigmaran'].sum()) == 0, 'range_window_exact': math.isclose(RADAR_RANGE_MAX_KM - RADAR_RANGE_MIN_KM, 57.411, abs_tol=1e-09)}
failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise RuntimeError(f'Final QA failed: {failed}')
top_actions = {}
for _, _, key, display_label in THRESHOLDS:
    top = action_summary.nlargest(5, f'window_flux_gt_{key}_m2_yr')
    top_actions[key] = [{'rank': rank, 'action_id': row.action_id, 'azimuth_deg': float(row.azimuth_deg), 'elevation_deg': float(row.elevation_deg), 'window_flux_m2_yr': float(row[f'window_flux_gt_{key}_m2_yr']), 'ratio_to_center': float(row[f'window_ratio_to_center_gt_{key}'])} for rank, (_, row) in enumerate(top.iterrows(), start=1)]
manifest = {'created_utc': datetime.now(timezone.utc).isoformat(), 'purpose': 'Directional ORDEM response/action library for later Bayesian adaptive-observation experiments', 'inputs': {'batch_zip': BATCH_NAME, 'batch_sha256': batch_hash, 'batch_completed_jobs': 80, 'baseline_zip': BASELINE_NAME, 'baseline_sha256': baseline_hash, 'baseline_completed_jobs': 1}, 'configuration': {'year': 2018, 'sensor_latitude_deg_from_output': float(inventory.sensor_lat_header_deg.iloc[0]), 'igloo_file': 'ALT_5_LEO.TIG', 'azimuth_grid_deg': [float(x) for x in sorted(inventory.azimuth_deg.unique())], 'elevation_grid_deg': [float(x) for x in sorted(inventory.elevation_deg.unique())], 'grid_spacing_deg': 2.5, 'center_pointing_deg': {'azimuth': 90.0, 'elevation': 75.0}, 'radar_cfar_valid_slant_range_km': [RADAR_RANGE_MIN_KM, RADAR_RANGE_MAX_KM]}, 'uncertainty_scope': {'directional_central_flux_estimates': 81, 'jobs_with_nasa_sigmaran_and_sigmapop': 1, 'interpretation': 'Only the original center job supplied uncertainty files. These are stored separately and must be used as a shared calibration/hyperprior assumption, not as 80 independently computed directional uncertainties.'}, 'row_counts': {'action_inventory': int(len(inventory)), 'wide_flux': int(len(wide_all)), 'action_response': int(len(response_all)), 'center_uncertainty_long': int(len(center_uncertainty))}, 'quality_checks': checks, 'top_actions_by_size_threshold': top_actions, 'limitations': ['ORDEM flux is a population-model prior, not a measured count and not a catalog identity.', 'The action grid is a local 20-degree by 20-degree neighborhood, not global steering.', 'Absolute expected radar counts require the empirical detector/selection model and exposure scale in the next notebook.', 'The completed EISCAT GMF data cover only the frozen 57.411 km slant-range window used here.']}
readme = f"research ORDEM 81-pointing action library\n=======================================\n\nSUCCESS: 80 batch pointings plus the original center pointing passed all structural and numeric checks.\n\nWhat this contains\n------------------\n- Central ORDEM 3.2 flux estimates for 81 local pointing actions.\n- The exact response matrix inside the completed radar's {RADAR_RANGE_MIN_KM:.3f}-{RADAR_RANGE_MAX_KM:.3f} km CFAR-valid slant-range window.\n- The original center job's NASA range-randomness and population uncertainties.\n- Directional heatmaps and a reproducibility manifest.\n\nWhat this does NOT claim\n------------------------\n- ORDEM flux is not an observed object count.\n- A high-flux pointing is not automatically the best adaptive pointing; information gain also depends on what is already known and on detector sensitivity.\n- The 80 batch jobs did not include separate uncertainty files. Center uncertainty is preserved as a shared calibration input, not relabeled as per-direction NASA output.\n- Nothing here identifies trajectories or creates collision warnings.\n\nNext analysis\n-------------\nUse ordem_action_response_matrix.csv.gz as the observation operator in the Bayesian notebook. Calibrate its absolute count scale with the frozen CA-CFAR/catalog dataset, then compare fixed-center, random, flux-greedy, and posterior-information-gain schedules under equal observing time.\n"
(OUTPUT_DIR / 'ordem_action_library_summary.json').write_text(json.dumps(manifest, indent=2))
(OUTPUT_DIR / 'README_ordem_action_library.txt').write_text(readme)
(OUTPUT_DIR / '_SUCCESS.json').write_text(json.dumps({'status': 'complete', 'created_utc': manifest['created_utc'], 'batch_sha256': batch_hash, 'baseline_sha256': baseline_hash, 'actions': 81, 'quality_checks': checks}, indent=2))
print('\n' + '=' * 72)
print('SUCCESS — ORDEM ACTION LIBRARY COMPLETE')
print('=' * 72)
print('Actions: 81 (80 batch + 1 center)')
print('Response rows:', f'{len(response_all):,}')
print('Center uncertainty rows:', f'{len(center_uncertainty):,}')
print('Output folder:', OUTPUT_DIR)
print('\nTop three radar-window actions for >1 mm:')
display(action_summary.nlargest(3, 'window_flux_gt_1mm_m2_yr')[['action_id', 'azimuth_deg', 'elevation_deg', 'window_flux_gt_1mm_m2_yr', 'window_ratio_to_center_gt_1mm']])
print('\nDo not rerun ORDEM. The next notebook is the Bayesian policy comparison.')


## Outputs
